In [1]:
#!/usr/bin/env python3
from datetime import datetime
import gymnasium as gym
import rospy 
import cv2
from cv_bridge import CvBridge, CvBridgeError
from sensor_msgs.msg import Image,Imu,LaserScan
from geometry_msgs.msg import Twist,Vector3
import queue
from gymnasium.spaces import Dict,Box
import numpy as np
from queue import Queue
import time
from jaxrl2.data import VariableCapacityBuffer

In [2]:
IMAGE_TOPIC="/camera/image_raw"
IMU_TOPIC="/imu/data_raw"
ROBOT_CMD_TOPIC="/cmd_vel"
LIDAR_TOPIC="/scan"
RATE = 60

In [3]:
def ros_vector3_to_np_array(msg):
    return np.array([msg.x,-1*msg.y,msg.z])

In [4]:
bridge=CvBridge()
image_size=64

# Define Queue
image_queue=Queue()
imu_queue=Queue()
lidar_queue=Queue()
action_queue=Queue()


#
image_space = Box(
            low=-1.0,
            high=1.0,
            shape=(image_size, image_size, 3,),
            dtype=np.float32,
        )
        
vec_space = Box(
    low=-5.1,
    high=5.1,
    shape=(4,),
    dtype=np.float32,
)

observation_space=Dict({"pixels":image_space, "vector":vec_space})

action_space = Box(
    low=np.array([-1.0, -1.0]),  # [steering, throttle/brake]
    high=np.array([1.0, 1.0]),
    dtype=np.float32
)
def image_callback(image):
    # print("image")
    image = bridge.imgmsg_to_cv2(image, "rgb8")
    image=cv2.resize(image,(image_size,image_size))
    image_queue.put(image_queue)

def imu_callback(imu):
    payload=dict(w=ros_vector3_to_np_array(imu.angular_velocity),a=ros_vector3_to_np_array(imu.linear_acceleration))
    imu_queue.put(payload)

def lidar_callback(scan):
    lidar_queue.put(scan)

def action_callback(action):
    action_queue.put(action)



/root/.virtualenvs/carla-rl-JfCMuBLH/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/root/.virtualenvs/carla-rl-JfCMuBLH/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
rospy.init_node("DATA_FILTER", anonymous=False)
image_sub = rospy.Subscriber(IMAGE_TOPIC,Image,image_callback)
lidar_sub = rospy.Subscriber(LIDAR_TOPIC,LaserScan,lidar_callback)
imu_sub = rospy.Subscriber(IMU_TOPIC,Imu,imu_callback)
robot_cmd =rospy.Subscriber(ROBOT_CMD_TOPIC, Twist, queue_size=10)
time.sleep(1.0)
replay_buffer = VariableCapacityBuffer(
        observation_space,
        action_space)
step=0
#put everything in a list becuase we need some metrics
a=[]
w=[]
scans=[]
images=[]
actions=[]
while True:
    # continue
    # time.sleep(2.0)
    print("image",image_queue.qsize())
    image=image_queue.get()
    imu=imu_queue.get()
    scan=lidar_queue.get()
    action=action_queue.get()

    # print("Image",image)
    a.append(imu.get("a"))
    w.append(imu.get("w"))
    scans.append(scan)
    actions.append(action)  
    images.append(image)
    # time.sleep(2)

    if (image_queue.empty() or imu_queue.empty() or lidar_queue.empty()):
        break

    # more data
    # image_queue.put(image)
    # imu=imu_queue.get(imu)
    # scan=lidar_queue.put(scan)
    # step+=1




image 0


In [ ]:
from scipy.signal import filtfilt, butter

def estimate_orientation(a, w, angle,dt, alpha=0.9, g_ref=(0., 0., 1.), theta_min=1e-6, highpass=.01, lowpass=.05):
    """
    Source:https://gist.github.com/phausamann/721fa3df0f8ef6f4f6f24b86fdde53c0
    """

    g_ref = np.array(g_ref)
    w = filtfilt(*butter(5, highpass, btype='high'), w, axis=0)
    w[np.linalg.norm(w, axis=1) < theta_min] = 0
    a = filtfilt(*butter(5, lowpass, btype='low'), a, axis=0)
    angle = (1-alpha)*(angle + w * dt) + (alpha)*(a)

    return angle

In [ ]:
import matplolib.pyplot as plt
x=np.array(range(len(a)))
plt.plot(x,a,label="Acceleration")
plt.plot(x,w,label="Angular Velocity")